<a href="https://colab.research.google.com/github/mvenyidonny/MyFiles/blob/main/ipynb_merger_webapp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Jupyter Notebook Merger</title>
    <!-- Tailwind CSS CDN -->
    <script src="https://cdn.tailwindcss.com"></script>
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap');
        body {
            font-family: 'Inter', sans-serif;
            background-color: #f0f2f5;
            display: flex;
            justify-content: center;
            align-items: flex-start; /* Align to top for better content display */
            min-height: 100vh;
            padding: 20px;
            box-sizing: border-box;
        }
        .container {
            background-color: #ffffff;
            border-radius: 12px;
            box-shadow: 0 4px 12px rgba(0, 0, 0, 0.1);
            padding: 24px;
            width: 100%;
            max-width: 700px;
            margin-top: 50px; /* Space from top */
        }
        .file-input-wrapper {
            position: relative;
            overflow: hidden;
            display: inline-block;
            width: 100%;
        }
        .file-input-wrapper input[type=file] {
            font-size: 100px;
            position: absolute;
            left: 0;
            top: 0;
            opacity: 0;
            cursor: pointer;
        }
        .btn-upload {
            background-color: #4CAF50; /* Green */
            color: white;
            padding: 12px 20px;
            border-radius: 8px;
            cursor: pointer;
            text-align: center;
            display: block;
            width: 100%;
            transition: background-color 0.3s ease;
        }
        .btn-upload:hover {
            background-color: #45a049;
        }
        .btn-download {
            background-color: #2196F3; /* Blue */
            color: white;
            padding: 12px 20px;
            border-radius: 8px;
            text-align: center;
            display: none; /* Hidden by default */
            width: 100%;
            margin-top: 16px;
            text-decoration: none; /* Remove underline for link */
            transition: background-color 0.3s ease;
        }
        .btn-download:hover {
            background-color: #0b7dda;
        }
        .loading-spinner {
            border: 4px solid rgba(0, 0, 0, 0.1);
            border-top: 4px solid #3498db;
            border-radius: 50%;
            width: 30px;
            height: 30px;
            animation: spin 1s linear infinite;
            display: none; /* Hidden by default */
            margin: 20px auto;
        }
        @keyframes spin {
            0% { transform: rotate(0deg); }
            100% { transform: rotate(360deg); }
        }
        .message {
            padding: 12px;
            border-radius: 8px;
            margin-top: 16px;
            font-weight: 500;
            text-align: center;
        }
        .message.success {
            background-color: #d4edda;
            color: #155724;
        }
        .message.error {
            background-color: #f8d7da;
            color: #721c24;
        }
        .message.warning {
            background-color: #fff3cd;
            color: #856404;
        }
        .file-list {
            margin-top: 16px;
            background-color: #f9f9f9;
            border: 1px solid #e0e0e0;
            border-radius: 8px;
            padding: 12px;
            max-height: 150px;
            overflow-y: auto;
        }
        .file-list-item {
            padding: 6px 0;
            border-bottom: 1px dashed #e9e9e9;
            font-size: 0.9em;
            color: #555;
        }
        .file-list-item:last-child {
            border-bottom: none;
        }
    </style>
</head>
<body class="bg-gray-100 flex items-center justify-center min-h-screen p-4">

    <div class="container p-8 space-y-6">
        <h1 class="text-3xl font-bold text-center text-gray-800 mb-6">Jupyter Notebook Merger</h1>

        <div class="file-input-wrapper">
            <button class="btn-upload flex items-center justify-center space-x-2">
                <svg xmlns="http://www.w3.org/2000/svg" class="h-6 w-6" fill="none" viewBox="0 0 24 24" stroke="currentColor">
                    <path stroke-linecap="round" stroke-linejoin="round" stroke-width="2" d="M4 16v1a3 3 0 003 3h10a3 3 0 003-3v-1m-4-8l-4-4m0 0L8 8m4-4v12" />
                </svg>
                <span>Select .ipynb Files</span>
            </button>
            <input type="file" id="notebookInput" multiple accept=".ipynb">
        </div>

        <div id="fileList" class="file-list hidden">
            <h3 class="text-lg font-semibold text-gray-700 mb-2">Selected Files:</h3>
            <ul id="selectedFilesList"></ul>
        </div>

        <button id="mergeButton" class="btn-download bg-purple-600 hover:bg-purple-700 text-white font-semibold py-3 px-6 rounded-lg transition-colors duration-200 ease-in-out cursor-pointer shadow-md" style="display: block;">
            Merge & Download Notebooks
        </button>

        <a id="downloadLink" class="btn-download">
            Click to Download Merged Notebook
        </a>

        <div id="loadingSpinner" class="loading-spinner"></div>
        <div id="messageArea" class="message hidden"></div>
    </div>

    <script>
        const notebookInput = document.getElementById('notebookInput');
        const mergeButton = document.getElementById('mergeButton');
        const downloadLink = document.getElementById('downloadLink');
        const loadingSpinner = document.getElementById('loadingSpinner');
        const messageArea = document.getElementById('messageArea');
        const fileListContainer = document.getElementById('fileList');
        const selectedFilesList = document.getElementById('selectedFilesList');

        // Define a template for a new Jupyter notebook structure (nbformat v4)
        // This ensures the output .ipynb file is correctly formatted.
        const newNotebookTemplate = {
            "cells": [],
            "metadata": {
                "kernelspec": {
                    "display_name": "Python 3",
                    "language": "python",
                    "name": "python3"
                },
                "language_info": {
                    "codemirror_mode": {
                        "name": "ipython",
                        "version": 3
                    },
                    "file_extension": ".py",
                    "mimetype": "text/x-python",
                    "name": "python",
                    "nbconvert_exporter": "python",
                    "pygments_lexer": "ipython3",
                    "version": "3.x"
                }
            },
            "nbformat": 4,
            "nbformat_minor": 5
        };

        // Helper function to create a new markdown cell for separation
        function createMarkdownCell(source_text) {
            return {
                "cell_type": "markdown",
                "metadata": {},
                "source": [source_text]
            };
        }

        // Helper to read file content as text
        function readFileAsText(file) {
            return new Promise((resolve, reject) => {
                const reader = new FileReader();
                reader.onload = (e) => resolve(e.target.result);
                reader.onerror = (e) => reject(new Error(`Failed to read file: ${e.target.error.code}`));
                reader.readAsText(file);
            });
        }

        // UI update functions
        function setLoading(isLoading) {
            if (isLoading) {
                loadingSpinner.style.display = 'block';
                mergeButton.disabled = true;
                downloadLink.style.display = 'none';
                messageArea.classList.add('hidden');
            } else {
                loadingSpinner.style.display = 'none';
                mergeButton.disabled = false;
            }
        }

        function showMessage(msg, type) {
            messageArea.textContent = msg;
            messageArea.className = `message ${type}`;
            messageArea.classList.remove('hidden');
        }

        function clearMessages() {
            messageArea.classList.add('hidden');
            messageArea.textContent = '';
        }

        // Event listener for file selection
        notebookInput.addEventListener('change', (event) => {
            clearMessages();
            downloadLink.style.display = 'none'; // Hide download link if new files are selected
            selectedFilesList.innerHTML = ''; // Clear previous list

            const files = event.target.files;
            if (files.length > 0) {
                fileListContainer.classList.remove('hidden');
                for (let i = 0; i < files.length; i++) {
                    const listItem = document.createElement('li');
                    listItem.textContent = files[i].name;
                    listItem.className = 'file-list-item';
                    selectedFilesList.appendChild(listItem);
                }
                showMessage(`Selected ${files.length} file(s). Click 'Merge & Download' to proceed.`, 'success');
            } else {
                fileListContainer.classList.add('hidden');
                showMessage('No files selected.', 'warning');
            }
        });

        // Event listener for the Merge & Download button
        mergeButton.addEventListener('click', async () => {
            const files = notebookInput.files;
            if (files.length === 0) {
                showMessage("Please select at least one Jupyter notebook file first.", 'warning');
                return;
            }

            setLoading(true);
            let allCells = [];
            let mergedNotebook = JSON.parse(JSON.stringify(newNotebookTemplate)); // Deep copy the template

            try {
                for (let i = 0; i < files.length; i++) {
                    const file = files[i];
                    const fileContent = await readFileAsText(file);
                    const notebook = JSON.parse(fileContent);

                    // Add a separator markdown cell before each notebook's content
                    allCells.push(createMarkdownCell(`## Content from: ${file.name}\n`));

                    // Add all cells from the current notebook
                    if (notebook.cells && Array.isArray(notebook.cells)) {
                        allCells.push(...notebook.cells);
                    } else {
                        showMessage(`Warning: Notebook '${file.name}' has no 'cells' property or it's not an array. Skipping its content.`, 'warning');
                    }
                }

                mergedNotebook.cells = allCells;
                const mergedJsonString = JSON.stringify(mergedNotebook, null, 2); // Pretty print JSON output

                // Create a Blob and URL for downloading the merged notebook
                const blob = new Blob([mergedJsonString], { type: 'application/json' });
                const url = URL.createObjectURL(blob);

                downloadLink.href = url;
                downloadLink.download = 'merged_notebook.ipynb'; // Default download filename
                downloadLink.style.display = 'block'; // Show the download link

                showMessage('Notebooks merged successfully! Click the download link above.', 'success');

            } catch (error) {
                console.error("Merge error:", error);
                showMessage(`An error occurred during merging: ${error.message}. Please check your files.`, 'error');
                downloadLink.style.display = 'none'; // Ensure download link is hidden on error
            } finally {
                setLoading(false); // Always hide loading spinner
            }
        });

    </script>
</body>
</html>